In [1]:
from collections import namedtuple
from time import sleep
import logging

from pulao.logging import init_logging

import pandas as pd

from pulao.swing import SwingManager
from pulao.bar import SBar, SBarManager, CBarManager
from vnpy.trader.constant import Exchange, Interval
from vnpy.trader.object import BarData
from pulao.trend import TrendManager

import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pulao.constant import *

from pulao.logging import logger

init_logging(level=logging.DEBUG)
open("./logs/pulao.log", "w").close()  # 每次运行前清空日志
logger.debug("开始运行")

df = pl.read_csv("../dataset/I8888.XDCE_60m.csv", try_parse_dates=True)
df = df.slice(0)  # test

sbar_list = []
columns = df.columns

for idx, row in enumerate(df.iter_rows()):
    row_dict = dict(zip(columns, row))
    # datetime,open,close,high,low,volume,money,open_interest,signal
    datetime = row_dict["datetime"]
    o = row_dict["open"]
    close = row_dict["close"]
    high = row_dict["high"]
    low = row_dict["low"]
    volume = row_dict["volume"]
    money = row_dict["money"]
    open_interest = row_dict["open_interest"]

    bar = BarData(
        gateway_name="ctp-test",
        symbol="i8888",
        exchange=Exchange.SHFE,
        interval=Interval.MINUTE,
        datetime=datetime,
        open_price=o,
        close_price=close,
        high_price=high,
        low_price=low,
        volume=volume,
        open_interest=open_interest,
        turnover=money,
    )
    sbar = SBar(
        symbol=bar.symbol,
        exchange=bar.exchange.value,
        interval=bar.interval.value,
        datetime=datetime,
        turnover=money,
        open_price=o,
        close_price=close,
        high_price=high,
        low_price=low,
        volume=volume,
    )

    sbar_list.append(sbar)
# 模拟行情数据接收
sbar_manager = SBarManager()
cbar_manager = CBarManager(sbar_manager=sbar_manager)
swing_manager = SwingManager(cbar_manager=cbar_manager)
trend_manager = TrendManager(swing_manager=swing_manager)
# 流入模拟数据
for sbar in sbar_list:
    sbar_manager.append(sbar)

logger.debug("运行结束")
# sleep(3) # 等一会，让日志输出完


{"event": "开始运行", "level": "debug", "logger": "__main__", "time": "2025-11-30 02:10:44"}
{"cbar_count": 1, "event": "用于组成分形的cbar数量不够", "level": "warning", "logger": "__main__", "time": "2025-11-30 02:10:44"}
{"swing_list": null, "event": "波段数量不足3个，跳过", "level": "debug", "logger": "__main__", "time": "2025-11-30 02:10:44"}
{"swing_list": null, "event": "波段数量不足3个，跳过", "level": "debug", "logger": "__main__", "time": "2025-11-30 02:10:44"}
{"cbar_count": 2, "event": "用于组成分形的cbar数量不够", "level": "warning", "logger": "__main__", "time": "2025-11-30 02:10:44"}
{"swing_list": null, "event": "波段数量不足3个，跳过", "level": "debug", "logger": "__main__", "time": "2025-11-30 02:10:44"}
{"fractal": "Fractal(left=CBar(id=120708060439117824, sbar_start_id=120708060418146304, sbar_end_id=120708060434923520, high_price=942.2349853515625, low_price=935.9240112304688, created_at=datetime.datetime(2025, 11, 30, 2, 10, 44, 256000), fractal_type=0), middle=CBar(id=120708060455895040, sbar_start_id=12070806045589504

AttributeError: 'Pandas' object has no attribute 'cbar_start_id'

In [8]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pulao.constant import *

#
# region . Plotly - Cbar
#
df_cbar = cbar_manager.df_cbar.to_pandas()
df_cbar["datetime"] = pd.date_range("2025-01-01", periods=df_cbar.shape[0], freq="h")
df_cbar["open_price"] = df_cbar["high_price"]
df_cbar["close_price"] = df_cbar["low_price"]
df_cbar.insert(0, "index", range(len(df_cbar)))

# region 构建波段数据
df_swing = swing_manager.df_swing.to_pandas()
df_swing["start_datetime"] = pd.Series(dtype="datetime64[ns]")
df_swing["end_datetime"] = pd.Series(dtype="datetime64[ns]")
df_swing.insert(0, "index", range(len(df_swing)))

for i, row in enumerate(df_swing.itertuples()):
    df = df_cbar[
        (df_cbar["id"] >= row.cbar_start_id) & (df_cbar["id"] <= row.cbar_end_id)
    ]
    start_datetime = df.iloc[0]["datetime"]
    end_datetime = df.iloc[-1]["datetime"]
    df_swing.at[i, "start_datetime"] = start_datetime
    df_swing.at[i, "end_datetime"] = end_datetime

xs_swing = []
ys_swing = []
texts_swing = []
text_positions_swing = []
for i, row in enumerate(df_swing.itertuples()):
    xs_swing += [row.start_datetime, row.end_datetime, None]
    start_index = df_cbar[(df_cbar["id"] == row.cbar_start_id)]["index"].iloc[0]
    end_index = df_cbar[(df_cbar["id"] == row.cbar_end_id)]["index"].iloc[0]
    if row.direction == Direction.UP:
        start_price = df_cbar[(df_cbar["id"] == row.cbar_start_id)]["low_price"].iloc[0]
        end_price = df_cbar[(df_cbar["id"] == row.cbar_end_id)]["high_price"].iloc[0]
        text_positions_swing += ["bottom center", "top center", "middle center"]
    else:
        start_price = df_cbar[(df_cbar["id"] == row.cbar_start_id)]["high_price"].iloc[
            0
        ]
        end_price = df_cbar[(df_cbar["id"] == row.cbar_end_id)]["low_price"].iloc[0]
        text_positions_swing += ["top center", "bottom center", "middle center"]
    ys_swing += [start_price, end_price, None]
    texts_swing += [start_index, end_index, None]
# endregion

# region 构建趋势数据
df_trend = trend_manager.df_trend.to_pandas()
df_trend["start_datetime"] = pd.Series(dtype="datetime64[ns]")
df_trend["end_datetime"] = pd.Series(dtype="datetime64[ns]")
df_trend.insert(0, "index", range(len(df_trend)))

for i, row in enumerate(df_trend.itertuples()):
    df = df_swing[
        (df_swing["id"] >= row.swing_start_id) & (df_swing["id"] <= row.swing_end_id)
    ]
    start_datetime = df.iloc[0]["start_datetime"]
    end_datetime = df.iloc[-1]["end_datetime"]
    df_trend.at[i, "start_datetime"] = start_datetime
    df_trend.at[i, "end_datetime"] = end_datetime

xs_trend = []
ys_trend = []
texts_trend = []
text_positions_trend = []
for i, row in enumerate(df_trend.itertuples()):
    xs_trend += [row.start_datetime, row.end_datetime, None]
    print(row.swing_start_id,df_swing[(df_swing["id"] == row.swing_start_id)])
    start_index = df_swing[(df_swing["id"] == row.swing_start_id)]["index"].iloc[0]
    end_index = df_swing[(df_swing["id"] == row.swing_end_id)]["index"].iloc[0]
    if row.direction == Direction.UP:
        start_price = df_swing[(df_swing["id"] == row.swing_start_id)]["low_price"].iloc[0]
        end_price = df_swing[(df_swing["id"] == row.swing_end_id)]["high_price"].iloc[0]
        text_positions_trend += ["bottom center", "top center", "middle center"]
    else:
        start_price = df_swing[(df_swing["id"] == row.swing_start_id)]["high_price"].iloc[
            0
        ]
        end_price = df_swing[(df_swing["id"] == row.swing_end_id)]["low_price"].iloc[0]
        text_positions_trend += ["top center", "bottom center", "middle center"]
    ys_trend += [start_price, end_price, None]
    texts_trend += [start_index, end_index, None]
# endregion

fig = make_subplots(rows=1, cols=1)
fig.add_trace(
    go.Candlestick(
        x=df_cbar["datetime"],
        open=df_cbar["open_price"],
        high=df_cbar["high_price"],
        low=df_cbar["low_price"],
        close=df_cbar["close_price"],
        name="K线",
    ),
    row=1,
    col=1,
)

df_fractal_bottom = df_cbar[(df_cbar["fractal_type"] == FractalType.BOTTOM)]

fig.add_trace(
    go.Scatter(
        x=df_fractal_bottom["datetime"],
        y=df_fractal_bottom["low_price"] * 0.995,  # 放在K线最低价下方一点，不遮挡蜡烛
        mode="markers+text",
        name="swing point low",
        marker=dict(
            symbol="triangle-up",
            size=14,
            color="#1E90FF",
        ),
        text=df_fractal_bottom["index"],  # ← 添加序号
        textposition="bottom center",  # ← 文字位置
        textfont=dict(size=10, color="white"),
        hovertemplate="<b>波段低点</b><br>"
        + "时间: %{x}<br>"
        + "价格: %{y:.2f}<br>"
        + "<extra></extra>",
    ),
    row=1,
    col=1,
)


df_fractal_top = df_cbar[(df_cbar["fractal_type"] == FractalType.TOP)]

fig.add_trace(
    go.Scatter(
        x=df_fractal_top["datetime"],
        y=df_fractal_top["high_price"] * 1.005,  # 放在K线最高价上方一点
        mode="markers+text",
        name="swing point high",
        marker=dict(
            symbol="triangle-down",
            size=14,
            color="#FF4500",
        ),
        text=df_fractal_top["index"],  # ← 添加序号
        textposition="top center",  # ← 文字位置
        textfont=dict(size=10, color="white"),
        hovertemplate="<b>波段高点</b><br>"
        + "时间: %{x}<br>"
        + "价格: %{y:.2f}<br>"
        + "<extra></extra>",
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=xs_swing,
        y=ys_swing,
        name="swing segments",
        mode="lines+text",  # lines+text 支持显示文字
        line=dict(width=2, color="orange"),
        text=texts_swing,
        textposition=text_positions_swing,  # 文字位置
    )
)

fig.add_trace(
    go.Scatter(
        x=xs_trend,
        y=ys_trend,
        name="trend segments",
        mode="lines+text",  # lines+text 支持显示文字
        line=dict(width=2, color="red"),
        text=texts_trend,
        textposition=text_positions_trend,  # 文字位置
    )
)

title = "Swing Chart"
fig.update_layout(
    title=title,
    height=900,
    hovermode="x unified",  # X轴悬停联动虚线
)

fig.update_xaxes(
    showgrid=False,
)

fig.update_yaxes(
    showgrid=False,
)

fig.show()

# endregion

120708061961650176    index                  id       cbar_start_id         cbar_end_id  \
1      1  120708061961650176  120708061164732416  120708061923901440   

        sbar_start_id         sbar_end_id  high_price   low_price  direction  \
1  120708061126983680  120708061907124224  988.528015  906.143005         -1   

   is_completed              created_at start_datetime        end_datetime  
1          True 2025-11-30 02:10:44.619     2025-01-02 2025-01-03 04:00:00  
120708065853964288    index                  id       cbar_start_id         cbar_end_id  \
8      8  120708065853964288  120708065015103488  120708065820409856   

        sbar_start_id         sbar_end_id  high_price   low_price  direction  \
8  120708065015103488  120708065786855424     829.763  748.465027          1   

   is_completed              created_at      start_datetime  \
8          True 2025-11-30 02:10:45.547 2025-01-08 10:00:00   

         end_datetime  
8 2025-01-09 12:00:00  
120708066667659264   

IndexError: single positional indexer is out-of-bounds

In [14]:
df_swing[df_swing["id"]>120708078692728832]

,index,id,cbar_start_id,cbar_end_id,sbar_start_id,sbar_end_id,high_price,low_price,direction,is_completed,created_at,start_datetime,end_datetime
28,28,120708079237988352,120708078629814272,120708079191851008,120708078621425664,120708079183462400,834.158997,802.969971,1,True,2025-11-30 02:10:48.738,2025-01-25 03:00:00,2025-01-25 15:00:00
29,29,120708081033150464,120708079191851008,120708080978624512,120708079183462400,120708080974430208,834.158997,752.098022,-1,True,2025-11-30 02:10:49.166,2025-01-25 15:00:00,2025-01-27 11:00:00
30,30,120708081377083392,120708080978624512,120708081272225792,120708080974430208,120708081200922624,782.034973,752.098022,1,True,2025-11-30 02:10:49.248,2025-01-27 11:00:00,2025-01-27 17:00:00
31,31,120708081901371392,120708081272225792,120708081851039744,120708081200922624,120708081842651136,782.034973,739.611023,-1,True,2025-11-30 02:10:49.373,2025-01-27 17:00:00,2025-01-28 05:00:00
32,32,120708082727649280,120708081851039744,120708082681511936,120708081842651136,120708082639568896,793.218018,739.611023,1,True,2025-11-30 02:10:49.570,2025-01-28 05:00:00,2025-01-29 00:00:00
33,33,120708084870938624,120708082681511936,120708084828995584,120708082639568896,120708084808024064,793.218018,690.559998,-1,True,2025-11-30 02:10:50.081,2025-01-29 00:00:00,2025-01-31 02:00:00
34,34,120708087161028608,120708084828995584,120708087114891264,120708084808024064,120708087106502656,767.096008,690.559998,1,True,2025-11-30 02:10:50.628,2025-01-31 02:00:00,2025-02-02 08:00:00
35,35,120708088574509056,120708087114891264,120708088524177408,120708087106502656,120708088507400192,767.096008,662.242004,-1,True,2025-11-30 02:10:50.964,2025-02-02 08:00:00,2025-02-03 12:00:00
36,36,120708089610502144,120708088524177408,120708089551781888,120708088507400192,120708089505644544,712.046997,662.242004,1,True,2025-11-30 02:10:51.211,2025-02-03 12:00:00,2025-02-04 14:00:00
37,37,120708089853771776,120708089551781888,120708089799245824,120708089505644544,120708089790857216,712.046997,669.476990,-1,True,2025-11-30 02:10:51.269,2025-02-04 14:00:00,2025-02-04 20:00:00


In [1]:
from typing import List

# 测试swing 构建算法
from IPython.display import display
from pulao.logging import logger, init_logging
import logging
from time import sleep
from pulao.swing import Swing, SwingManager
from pulao.trend import Trend, TrendManager
from pulao.bar import CBar, CBarManager, SBarManager
from pulao.constant import *

init_logging(level=logging.DEBUG)
with open("./logs/pulao.log", "w"):
    pass  # 每次运行前清空日志

logger.debug("开始运行")


def format_df_trend():
    import polars as pl

    if "index" in swing_manager.df_swing.columns:
        swing_manager.df_swing = swing_manager.df_swing.drop("index")
    swing_manager.df_swing = swing_manager.df_swing.with_row_index("index")

    df_swing_tmp = swing_manager.df_swing.select(["id", "index"])
    df1 = trend_manager.df_trend.join(
        df_swing_tmp, left_on="swing_start_id", right_on="id", how="left"
    )
    df2 = trend_manager.df_trend.join(
        df_swing_tmp, left_on="swing_end_id", right_on="id", how="left"
    )
    df1 = df1.rename({"index": "swing_start_index"})
    df2 = df2.rename({"index": "swing_end_index"})
    df2 = df2.select(["id", "swing_end_index"])
    df1 = df1.join(df2, on="id", how="left")
    df = df1.select(
        [
            "id",
            "swing_start_index",
            "swing_end_index",
            "swing_start_id",
            "swing_end_id",
            "high_price",
            "low_price",
            "direction",
            "is_completed",
            "created_at",
        ]
    )

    return df


cbar_manager = CBarManager(sbar_manager=SBarManager())
swing_manager = SwingManager(cbar_manager=cbar_manager)
trend_manager = TrendManager(swing_manager=swing_manager)
df_cbar = cbar_manager.read_parquet()
df_swing = swing_manager.read_parquet()
swing_manager.df_swing = swing_manager.df_swing.slice(0, 0)  # 清空原数据
for i in range(df_swing.height - 1):
    swing = Swing(**df_swing.row(i, named=True))
    swing_manager._append_swing(swing)
    swing_manager.notify(EventType.SWING_CHANGED)


logger.debug("运行结束")
sleep(0.2)


display("df_trend")
df = format_df_trend()
display(df)

display("df_swing")
swing_manager.df_swing

{"event": "开始运行", "level": "debug", "logger": "__main__", "time": "2025-11-30 01:52:23"}
{"swing_list": ["Swing(id=120703444741783552, direction=1, cbar_start_id=120691400458108928, cbar_end_id=120691400957231104, sbar_start_id=120691400458108928, sbar_end_id=120691400898510848, high_price=988.5280151367188, low_price=930.2979736328125, is_completed=True, created_at=datetime.datetime(2025, 11, 30, 1, 52, 23, 788000))"], "event": "波段数量不足3个，跳过", "level": "debug", "logger": "__main__", "time": "2025-11-30 01:52:23"}
{"swing_list": ["Swing(id=120703444741783552, direction=1, cbar_start_id=120691400458108928, cbar_end_id=120691400957231104, sbar_start_id=120691400458108928, sbar_end_id=120691400898510848, high_price=988.5280151367188, low_price=930.2979736328125, is_completed=True, created_at=datetime.datetime(2025, 11, 30, 1, 52, 23, 788000))", "Swing(id=120703444750172160, direction=-1, cbar_start_id=120691400957231104, cbar_end_id=120691401749954560, sbar_start_id=120691400898510848, sba

'df_trend'

id,swing_start_index,swing_end_index,swing_start_id,swing_end_id,high_price,low_price,direction,is_completed,created_at
u64,u32,u32,u64,u64,f32,f32,i8,bool,datetime[ms]
120703444871806976,1,9,120703444750172160,120703444813086721,955.924988,731.429016,-1,true,2025-11-30 01:52:23.819
120703444951498752,10,16,120703444817281024,120703444855029760,923.666016,731.429016,1,true,2025-11-30 01:52:23.838
120703445073133568,17,39,120703444855029761,120703445031190528,923.666016,657.40802,-1,true,2025-11-30 01:52:23.867
120703445127659520,40,40,120703445047967744,120703445047967744,842.429993,657.40802,1,true,2025-11-30 01:52:23.880
120703445169602560,41,43,120703445056356352,120703445077327872,842.429993,732.270996,-1,true,2025-11-30 01:52:23.890
120703445173796864,44,56,120703445081522176,120703445165408256,835.335022,732.270996,1,false,2025-11-30 01:52:23.891


'df_swing'

index,id,cbar_start_id,cbar_end_id,sbar_start_id,sbar_end_id,high_price,low_price,direction,is_completed,created_at
u32,u64,u64,u64,u64,u64,f32,f32,i8,bool,datetime[ms]
0,120703444741783552,120691400458108928,120691400957231104,120691400458108928,120691400898510848,988.528015,930.297974,1,true,2025-11-30 01:52:23.788
1,120703444750172160,120691400957231104,120691401749954560,120691400898510848,120691401733177344,988.528015,906.143005,-1,true,2025-11-30 01:52:23.790
2,120703444771143680,120691401749954560,120691402085498880,120691401733177344,120691402085498880,955.924988,906.143005,1,true,2025-11-30 01:52:23.795
3,120703444779532288,120691402085498880,120691403138269184,120691402085498880,120691403138269184,955.924988,842.133972,-1,true,2025-11-30 01:52:23.797
4,120703444787920896,120691403138269184,120691403230543872,120691403138269184,120691403230543872,879.856995,842.133972,1,true,2025-11-30 01:52:23.799
…,…,…,…,…,…,…,…,…,…,…
52,120703445148631040,120691446557704192,120691447367204864,120691446545121280,120691447350427648,786.112,758.200989,1,true,2025-11-30 01:52:23.885
53,120703445152825344,120691447367204864,120691449116229632,120691447350427648,120691449091063808,786.112,741.731995,-1,true,2025-11-30 01:52:23.886
54,120703445152825345,120691449116229632,120691454233280512,120691449091063808,120691454216503296,824.000977,741.731995,1,true,2025-11-30 01:52:23.886


In [4]:
swing_manager.df_swing = swing_manager.df_swing.drop("index")
swing_manager.next_swing(120701055628476416)

Swing(id=120701055628476416, direction=-1, cbar_start_id=120691405457719296, cbar_end_id=120691406120419328, sbar_start_id=120691405440942080, sbar_end_id=120691406082670592, high_price=829.7630004882812, low_price=731.4290161132812, is_completed=True, created_at=datetime.datetime(2025, 11, 30, 1, 42, 54, 179000))

In [ ]:
# 可视化日志
import pandas as pd

df = pd.read_json("./logs/pulao.log", lines=True)
df = df.drop(columns=["logger", "level", "time"])
priority_cols = ["event"]
other_cols = [c for c in df.columns if c not in priority_cols]
df = df[priority_cols + other_cols]
df["info"] = df[other_cols].apply(lambda row: row.dropna().tolist(), axis=1)
df = df.drop(columns=other_cols)
df["info"] = df["info"].apply(lambda x: "" if x == [] else x)
df

In [ ]:
cbar_manager.read_parquet()